In [1]:
import os
import cv2
import numpy as np

# ==============================================================================
# 1. Função de Letterbox (Redimensionamento com preservação de Aspect Ratio)
# ==============================================================================
def letterbox(img, tamanho_alvo=(640, 640), cor_fundo=(114, 114, 114)):
    """
    Redimensiona e centraliza a imagem mantendo sua proporção original,
    preenchendo os espaços vazios com uma borda uniforme.
    """
    altura_orig, largura_orig = img.shape[:2]
    largura_alvo, altura_alvo = tamanho_alvo

    # 1.1 Calcula a taxa de escala que cabe perfeitamente no quadro alvo
    escala = min(largura_alvo / largura_orig, altura_alvo / altura_orig)
    nova_largura = int(round(largura_orig * escala))
    nova_altura = int(round(altura_orig * escala))

    # 1.2 Redimensiona a imagem com a escala calculada
    img_redim = cv2.resize(
        img, (nova_largura, nova_altura), interpolation=cv2.INTER_LINEAR
    )

    # 1.3 Calcula o padding (espaçamento) necessário nos eixos X e Y
    pad_w = largura_alvo - nova_largura
    pad_h = altura_alvo - nova_altura

    # Divide o preenchimento igualmente entre os dois lados
    top = pad_h // 2
    bottom = pad_h - top
    left = pad_w // 2
    right = pad_w - left

    # 1.4 Adiciona as bordas constantes
    img_letterbox = cv2.copyMakeBorder(
        img_redim,
        top,
        bottom,
        left,
        right,
        cv2.BORDER_CONSTANT,
        value=cor_fundo,
    )

    # Retorna a imagem formatada e os fatores para desmapear coordenadas no pós-processamento
    info_mapeamento = {"escala": escala, "pad_x": left, "pad_y": top}
    return img_letterbox, info_mapeamento


# ==============================================================================
# 2. Carregamento da Imagem Original
# ==============================================================================
caminho_imagem = "bentoespock.jpeg"
if not os.path.exists(caminho_imagem):
    caminho_imagem = "bentoespock_3.jpeg"

img_bgr = cv2.imread(caminho_imagem)
if img_bgr is None:
    raise FileNotFoundError(
        f"Não foi possível carregar a imagem '{caminho_imagem}'."
    )

print("--- 1. ENTRADA ORIGINAL (OpenCV) ---")
print(f"Tipo do dado original: {img_bgr.dtype}")
print(f"Dimensões originais (H, W, C): {img_bgr.shape}")
print(f"Intervalo de intensidade: [{img_bgr.min()}, {img_bgr.max()}]")

# ==============================================================================
# 3. Pipeline de Conversão: Da Matriz ao Tensor de IA
# ==============================================================================

# Passo A: Letterboxing para 640x640 (padrão YOLO)
img_box, info_pad = letterbox(img_bgr, tamanho_alvo=(640, 640))
cv2.imwrite("aula05_01_letterbox_visualizacao.jpg", img_box)

# Passo B: Conversão do espaço de cor BGR para RGB
img_rgb = cv2.cvtColor(img_box, cv2.COLOR_BGR2RGB)

# Passo C: Normalização e Conversão Numérica (uint8 [0, 255] -> float32 [0.0, 1.0])
tensor = img_rgb.astype(np.float32) / 255.0

# Passo D: Transposição de Eixos (H, W, C) -> (C, H, W)
# Eixo 2 (Canais) vai para frente, Eixo 0 (Altura) e Eixo 1 (Largura) seguem
tensor = np.transpose(tensor, (2, 0, 1))

# Passo E: Adição da Dimensão do Lote / Batch (C, H, W) -> (1, C, H, W)
tensor_batch = np.expand_dims(tensor, axis=0)

# ==============================================================================
# 4. Inspeção Estrutural do Tensor Final
# ==============================================================================
print("\n--- 2. TENSOR FINAL ESTRUTURADO (Pronto para a Rede Neural) ---")
print(f"Tipo do dado final: {tensor_batch.dtype}")
print(f"Dimensões do tensor final (Batch, Canais, H, W): {tensor_batch.shape}")
print(f"Intervalo de valores: [{tensor_batch.min():.4f}, {tensor_batch.max():.4f}]")
print(f"Memória contígua em C: {tensor_batch.flags['C_CONTIGUOUS']}")

# ==============================================================================
# 5. Exibição Visual da Imagem Preparada
# ==============================================================================
cv2.imshow("Original BGR", img_bgr)
cv2.imshow("Entrada Formatada com Letterbox (640x640)", img_box)
print("\nPressione qualquer tecla na janela aberta para encerrar...")
cv2.waitKey(0)
cv2.destroyAllWindows()

--- 1. ENTRADA ORIGINAL (OpenCV) ---
Tipo do dado original: uint8
Dimensões originais (H, W, C): (1280, 960, 3)
Intervalo de intensidade: [0, 204]

--- 2. TENSOR FINAL ESTRUTURADO (Pronto para a Rede Neural) ---
Tipo do dado final: float32
Dimensões do tensor final (Batch, Canais, H, W): (1, 3, 640, 640)
Intervalo de valores: [0.0000, 0.7490]
Memória contígua em C: False

Pressione qualquer tecla na janela aberta para encerrar...
